# Прогнозирование и обнаружение аномалий во временном ряду

**Датасет:** Numenta Anomaly Benchmark (NAB), `machine_temperature_system_failure.csv`.  
**Цель:** подготовить временной ряд температуры промышленной машины, сравнить базовый прогноз с SARIMA и обнаружить аномальные режимы двумя методами.

Источник: [официальный репозиторий NAB](https://github.com/numenta/NAB).

## 1. Предметная область и постановка задачи

Каждая строка содержит временную метку и показание температурного датчика. Прогноз помогает оценивать ожидаемую температуру и заранее замечать изменение режима работы. Аномалией считаем необычное отклонение или интервал, совпадающий с официальным окном NAB; на практике это может соответствовать сбою, перегреву либо смене рабочего режима.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.pipeline import (load_and_prepare, apply_reference_labels, forecast, detect_anomalies, save_figures)
pd.set_option('display.max_columns', 20)
print(ROOT)

/Users/nikiforova/programming/time_series_practice

## 2. Загрузка и первичный просмотр данных

In [2]:
raw = pd.read_csv(ROOT / 'data/raw/machine_temperature_system_failure.csv')
print(raw.head().to_string(index=False))
print(f'Форма: {raw.shape}')
print('Типы данных:', raw.dtypes.astype(str).to_dict())
print('Пропуски:', raw.isna().sum().to_dict())
print('Повторные временные метки:', raw.duplicated('timestamp').sum())

          timestamp     value
2013-12-02 21:15:00 73.967322
2013-12-02 21:20:00 74.935882
2013-12-02 21:25:00 76.124162
2013-12-02 21:30:00 78.140707
2013-12-02 21:35:00 79.329836
Форма: (22695, 2)
Типы данных: {'timestamp': 'object', 'value': 'float64'}
Пропуски: {'timestamp': 0, 'value': 0}
Повторные временные метки: 12

## 3. Подготовка данных

Метки времени преобразуются в `datetime`, строки сортируются. Для 12 повторных меток берётся среднее измерений: этот выбор сохраняет информацию и формирует однозначный индекс. Затем ряд приводится к частоте 5 минут. Разрывов и пропусков после этого нет; интерполяция оставлена в конвейере как воспроизводимая защита для возможного обновления файла. Случайное перемешивание не применяется.

In [3]:
clean, audit = load_and_prepare()
data, windows = apply_reference_labels(clean)
print(pd.Series(audit).to_string())
print(f'Официальных окон аномалий: {len(windows)}')

raw_rows                               22695
raw_columns                                2
missing_values                             0
duplicate_timestamps                      12
clean_rows                             22683
missing_after_reindex                      0
remaining_missing                          0
start                    2013-12-02 21:15:00
end                      2014-02-19 15:25:00
frequency                          5 minutes
Официальных окон аномалий: 4

## 4. Визуальный анализ

Ряд заметно нестационарен: уровень температуры меняется ступенчато, присутствуют резкие локальные скачки и продолжительные режимы с иным уровнем. Скользящие среднее и стандартное отклонение подтверждают изменение локального уровня и волатильности. Поэтому одного глобального порога недостаточно.

![Временной ряд](../results/figures/01_time_series.png)

![Скользящие статистики](../results/figures/02_rolling_statistics.png)

## 5. Прогнозирование

Ряд усредняется по часу. Первые 80% наблюдений образуют обучающую часть, последние 20% - тестовую; хронологический порядок сохранён. Базовая модель повторяет значение того же часа предыдущих суток. Дополнительная модель SARIMA(1,1,1)(1,0,0,24) учитывает динамику ошибок и суточный цикл.

In [4]:
forecasts, forecast_metrics, forecast_meta = forecast(data)
print(pd.Series(forecast_meta).to_string())
print('\nМетрики:')
print(forecast_metrics.round(3).to_string(index=False))

hourly_observations                   1891
train_observations                    1512
test_observations                      379
split_timestamp        2014-02-03 21:00:00
sarima_aic                      7673.31686

Метрики:
         model    MAE   RMSE  MAPE_percent
        sarima 11.838 22.321        26.090
seasonal_naive 13.494 22.328        19.882

![Сравнение прогнозов](../results/figures/03_forecast_comparison.png)

По RMSE лучшей стала **sarima**: 22.32 против 22.33 у базовой модели. Разница по RMSE мала, а по MAPE базовая модель лучше (19.88% против 26.09%). Следовательно, SARIMA немного лучше штрафует крупные ошибки, но не даёт устойчивого превосходства по всем метрикам. Это важное ограничение, а не повод объявлять модель безусловно лучшей.

## 6. Обнаружение аномалий

Реализованы два метода. Первый оценивает робастный z-score ошибки суточного прогноза с порогом 4.0. Второй - Isolation Forest с долей выбросов 3%; он использует температуру, абсолютный скачок, суточную ошибку и отклонение от локальной медианы. Для проверки точки сопоставляются с четырьмя официальными окнами NAB.

In [5]:
detected, anomaly_metrics, anomaly_meta = detect_anomalies(data, windows)
print(pd.Series(anomaly_meta).to_string())
print('\nМетрики:')
print(anomaly_metrics.round(3).to_string(index=False))

true_anomalous_points         2268.000000
true_anomaly_share_percent       9.998677
reference_windows                4.000000

Метрики:
           method  anomalous_points  precision  recall    F1  windows_detected  windows_total
 isolation_forest               681      0.608   0.183 0.281                 4              4
robust_seasonal_z               531      0.710   0.166 0.269                 2              4

![Обнаруженные аномалии](../results/figures/04_anomaly_detection.png)

По точечному F1 лучшим стал **isolation_forest**: F1=0.281, precision=0.608, recall=0.183. Он подал сигнал во всех 4 из 4 окон. Робастный метод точнее (0.710), но нашёл только 2 из 4 окон. Невысокий точечный recall ожидаем: NAB размечает продолжительные окна, а детектор отмечает прежде всего наиболее необычные точки внутри них.

## 7. Итоговые выводы

1. После усреднения повторных меток получен регулярный ряд из 22 683 пятиминутных наблюдений без пропусков.
2. SARIMA имеет минимально меньший RMSE, но сезонная базовая модель лучше по MAPE; усложнение модели не принесло однозначного выигрыша.
3. Isolation Forest обнаружил все четыре известных события, а робастный суточный детектор дал более высокую точность отдельных сигналов. Практически полезно объединять их: Isolation Forest использовать для охвата событий, статистический метод - для более консервативных предупреждений.
4. Ограничения: короткая история, единственный датчик, фиксированные пороги и точечная метрика для окон событий. Развитие проекта: подбор гиперпараметров только на валидационном отрезке, многомерные признаки, оценка задержки обнаружения и адаптивные пороги.

## 8. Сохранённые результаты

Таблицы находятся в `results/*.csv`, графики - в `results/figures/`, очищенный ряд - в `data/processed/`. Полный повторный запуск выполняется командой `python src/pipeline.py`.